In [ ]:
import os
import glob
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

# Global Pandas options to avoid scientific notation for Lat/Lon coordinates
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.5f' % x)

# Visualization styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [ ]:
# 1. Locate all Uber monthly CSV files with fallback paths
file_paths = sorted(glob.glob('data/uber-raw-data-*.csv'))
if len(file_paths) == 0:
    file_paths = sorted(glob.glob('../data/uber-raw-data-*.csv'))

# 2. Ingest files and strip whitespaces from column headers to prevent mismatch errors
df_list = []
for file in file_paths:
    temp_df = pd.read_csv(file)
    temp_df.columns = temp_df.columns.str.strip()
    df_list.append(temp_df)

# 3. Concatenate all datasets and remove duplicate records (e.g., 7,749 duplicates in April)
df_raw = pd.concat(df_list, axis=0, ignore_index=True, sort=False)
df_clean = df_raw.drop_duplicates().copy()

# 4. Construct GeoDataFrame with standard WGS84 spatial reference system (EPSG:4326)
gdf_uber = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean['Lon'], df_clean['Lat']),
    crs="EPSG:4326"
)